#### ⏰ Deadline: Wednesday, July 8, 20:00
### What is this notebook about?

You will train a small neural network on the rural screening data and export its predictions in two ways:

1. **Single prediction** per patient (standard inference, dropout off)
2. **Multiple predictions** per patient (MC Dropout — dropout stays on, many stochastic forward passes)

Both result in CSV files saved to `../data/`.

---
#### 🟢 Station 1 — Load the data

The dataset contains two features, a disease label, and an `lr_split` column that tells us which patients were used to train the *original* logistic model.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/rural_screening.csv")

print(f"Patients: {len(df)}")
print(f"Disease prevalence: {df['disease'].mean():.1%}")
print(df["lr_split"].value_counts())

Patients: 3000
Disease prevalence: 30.5%
lr_split
train         2400
validation     600
Name: count, dtype: int64


---
#### 🟢 Station 2 — Train the MLP

We train on `lr_split == "train"` only. The model uses **dropout layers** so we can later run MC Dropout at prediction time.

Architecture: `Linear(2,32) → ReLU → Dropout → Linear(32,16) → ReLU → Dropout → Linear(16,1)`

In [2]:
# --- Settings ---
SEED = 43
MC_SAMPLES = 50   # how many stochastic predictions per patient (MC Dropout)
DROPOUT_P = 0.2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

FEATURE_COLS = ["feature 1", "feature 2"]

# --- Split using the lr_split column from the CSV ---
train_idx = df.index[df["lr_split"] == "train"]
holdout_idx = df.index[df["lr_split"] == "validation"]

X_train = df.loc[train_idx, FEATURE_COLS].to_numpy(np.float32)
y_train = df.loc[train_idx, "disease"].to_numpy(np.float32)

# Standardize: learn mean/std from training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)

# Standardize all patients (for later prediction on the full dataset)
X_all = scaler.transform(df[FEATURE_COLS].to_numpy(np.float32)).astype(np.float32)
y_all = df["disease"].to_numpy(np.float32)
X_holdout = X_all[df.index.get_indexer(holdout_idx)]
y_holdout = df.loc[holdout_idx, "disease"].to_numpy(np.float32)

print(f"Training patients: {len(train_idx)}")
print(f"Hold-out patients: {len(holdout_idx)}")


# --- Define the MLP (with dropout for MC Dropout later) ---
class MLP(nn.Module):
    def __init__(self, n_features, dropout_p=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)  # raw logits, shape (n_patients,)


model = MLP(len(FEATURE_COLS), dropout_p=DROPOUT_P)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)

# --- Train for 100 epochs ---
for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

print("Training finished (100 epochs)")

Training patients: 2400
Hold-out patients: 600
Training finished (100 epochs)


---
#### 🟢 Station 3 — Predict and save CSV files

We predict on **all patients** and save two CSV files to `../data/`:

| File | Contents |
|------|----------|
| `MLP-single-predictions.csv` | one prediction per patient |
| `MLP-multiple-predictions.csv` | `MC_SAMPLES` predictions per patient (`prediction_1` … `prediction_50`) |

In [3]:
DATA_DIR = Path("../data")


def predict_single(model, X):
    """One prediction per patient: dropout OFF, sigmoid(logits)."""
    model.eval()
    with torch.no_grad():
        X_t = torch.from_numpy(X.astype(np.float32))
        return torch.sigmoid(model(X_t)).numpy()


def predict_mc_samples(model, X, n_samples=50):
    """MC Dropout: dropout stays ON, return all stochastic predictions.
    Shape: (n_samples, n_patients)."""
    X_t = torch.from_numpy(X.astype(np.float32))
    model.train()  # keeps dropout active
    samples = []
    with torch.no_grad():
        for _ in range(n_samples):
            samples.append(torch.sigmoid(model(X_t)).numpy())
    model.eval()
    return np.stack(samples)


# --- Single predictions for all patients ---
single_probs = predict_single(model, X_all)

single_df = pd.DataFrame({
    "feature 1": df["feature 1"].to_numpy(),
    "feature 2": df["feature 2"].to_numpy(),
    "prediction": single_probs,
    "disease": df["disease"].to_numpy(),
})

single_path = DATA_DIR / "MLP-single-predictions.csv"
single_df.to_csv(single_path, index=False)
print(f"Saved {single_path} ({len(single_df)} patients)")

# Quick check on hold-out set
holdout_single = single_probs[df.index.get_indexer(holdout_idx)]
print(f"Hold-out Brier score (single): {brier_score_loss(y_holdout, holdout_single):.4f}")


# --- Multiple MC Dropout predictions for all patients ---
mc_samples = predict_mc_samples(model, X_all, n_samples=MC_SAMPLES)
# mc_samples shape: (50, n_patients) → one column per sample

multiple_df = pd.DataFrame({
    "feature 1": df["feature 1"].to_numpy(),
    "feature 2": df["feature 2"].to_numpy(),
    "disease": df["disease"].to_numpy(),
})
for i in range(MC_SAMPLES):
    multiple_df[f"prediction_{i + 1}"] = mc_samples[i]

multiple_path = DATA_DIR / "MLP-multiple-predictions.csv"
multiple_df.to_csv(multiple_path, index=False)
print(f"Saved {multiple_path} ({len(multiple_df)} patients × {MC_SAMPLES} predictions)")

# MC mean on hold-out (for comparison)
holdout_mc_mean = mc_samples[:, df.index.get_indexer(holdout_idx)].mean(axis=0)
print(f"Hold-out Brier score (MC mean): {brier_score_loss(y_holdout, holdout_mc_mean):.4f}")

display(single_df.head())
display(multiple_df.head())

Saved ../data/MLP-single-predictions.csv (3000 patients)
Hold-out Brier score (single): 0.1279
Saved ../data/MLP-multiple-predictions.csv (3000 patients × 50 predictions)
Hold-out Brier score (MC mean): 0.1293


,feature 1,feature 2,prediction,disease
0,0.244230,-1.402147,0.631124,1
1,0.678178,0.409495,0.333893,1
2,-0.585529,0.294501,0.180099,0
3,-0.908673,1.303086,0.103135,0
4,-1.991838,-1.347910,0.128141,0


,feature 1,feature 2,disease,prediction_1,prediction_2,prediction_3,prediction_4,prediction_5,prediction_6,prediction_7,...,prediction_41,prediction_42,prediction_43,prediction_44,prediction_45,prediction_46,prediction_47,prediction_48,prediction_49,prediction_50
0,0.244230,-1.402147,1,0.762074,0.528377,0.803250,0.476477,0.427847,0.778423,0.458542,...,0.544502,0.820401,0.705683,0.725694,0.704255,0.546389,0.681451,0.667906,0.527261,0.678332
1,0.678178,0.409495,1,0.272327,0.293524,0.312734,0.398982,0.293647,0.373205,0.351959,...,0.416284,0.299943,0.368006,0.252397,0.365615,0.397567,0.470774,0.352867,0.370937,0.422159
2,-0.585529,0.294501,0,0.148239,0.230736,0.205048,0.182701,0.196809,0.159047,0.263664,...,0.152853,0.208669,0.220273,0.165438,0.149072,0.109431,0.188739,0.154579,0.143666,0.130038
3,-0.908673,1.303086,0,0.143763,0.220322,0.093505,0.312721,0.039426,0.154473,0.129298,...,0.087234,0.176458,0.177414,0.065940,0.115566,0.149800,0.104830,0.042091,0.140229,0.079647
4,-1.991838,-1.347910,0,0.141650,0.161990,0.052158,0.208912,0.168560,0.110797,0.140685,...,0.080779,0.125660,0.145741,0.108388,0.145437,0.056308,0.215342,0.257607,0.051109,0.303210
